# Amazon Sales Analysis — Python EDA
**Dataset:** Amazon_Combined_Data.xlsx | **Records:** 89,082 | **Period:** Jan 2019 – Dec 2022  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Goal:** Explore product categories, pricing, reviews, and order trends to derive business insights


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print("Libraries loaded successfully")

## 1. Load & Inspect Data

In [ ]:
df = pd.read_excel('Amazon_Combined_Data.xlsx')

# Clean column names
df.columns = ['Category', 'Description', 'Price', 'Reviews', 'Shipment', 'Order_Date']

# Strip whitespace from category
df['Category'] = df['Category'].str.strip()

print(f"Shape: {df.shape}")
print(f"Date range: {df['Order_Date'].min().date()} to {df['Order_Date'].max().date()}")
df.head()

## 2. Data Quality Check

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Null Values ===")
print(df.isnull().sum())
print()
print("=== Duplicates ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Remove products with Price = 0 (data entry errors)
df = df[df['Price'] > 0]

# Extract time features
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Quarter'] = df['Order_Date'].dt.quarter

# Price segmentation
def price_segment(price):
    if price < 25:
        return 'Budget (<$25)'
    elif price <= 100:
        return 'Mid-range ($25-$100)'
    else:
        return 'Premium (>$100)'

df['Price_Segment'] = df['Price'].apply(price_segment)

print(f"Clean dataset shape: {df.shape}")
print(f"\nPrice segments:")
print(df['Price_Segment'].value_counts())

## 4. Product Category Analysis

In [ ]:
cat_summary = df.groupby('Category').agg(
    Total_Products=('Description', 'count'),
    Avg_Price=('Price', 'mean'),
    Total_Reviews=('Reviews', 'sum'),
    Avg_Reviews=('Reviews', 'mean')
).round(2).sort_values('Total_Products', ascending=False)

print(cat_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Product count by category
cat_counts = df['Category'].value_counts()
bars = axes[0].barh(cat_counts.index, cat_counts.values, color=sns.color_palette('Blues_r', len(cat_counts)))
axes[0].set_title('Total Products by Category')
axes[0].set_xlabel('Number of Products')
for bar, val in zip(bars, cat_counts.values):
    axes[0].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)

# Chart 2: Average price by category
avg_price = df.groupby('Category')['Price'].mean().sort_values(ascending=True)
bars2 = axes[1].barh(avg_price.index, avg_price.values, color=sns.color_palette('Oranges_r', len(avg_price)))
axes[1].set_title('Average Price by Category ($)')
axes[1].set_xlabel('Average Price (USD)')
for bar, val in zip(bars2, avg_price.values):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'${val:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart1_category_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Insight: Men Shoes and Men Clothes dominate volume. Laptop has the highest avg price at ~$1,001.")

## 5. Order Trend Analysis (2019–2022)

In [ ]:
yearly = df.groupby('Year').size().reset_index(name='Orders')
monthly = df.groupby(['Year', 'Month', 'Month_Name']).size().reset_index(name='Orders')
monthly_2022 = monthly[monthly['Year'] == 2022].sort_values('Month')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Yearly orders
axes[0].bar(yearly['Year'].astype(str), yearly['Orders'],
            color=['#aec6e8','#6baed6','#3182bd','#08519c'])
axes[0].set_title('Total Orders by Year')
axes[0].set_ylabel('Number of Orders')
for i, (yr, cnt) in enumerate(zip(yearly['Year'], yearly['Orders'])):
    axes[0].text(i, cnt + 200, f'{cnt:,}', ha='center', fontsize=10, fontweight='bold')

# 2022 monthly trend
axes[1].plot(monthly_2022['Month_Name'], monthly_2022['Orders'],
             marker='o', linewidth=2.5, color='#3182bd', markersize=7)
axes[1].fill_between(range(len(monthly_2022)), monthly_2022['Orders'], alpha=0.15, color='#3182bd')
axes[1].set_title('Monthly Order Trend — 2022')
axes[1].set_ylabel('Orders')
axes[1].set_xticks(range(len(monthly_2022)))
axes[1].set_xticklabels(monthly_2022['Month_Name'], rotation=45)

plt.tight_layout()
plt.savefig('chart2_order_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("Insight: Orders grew 50% from 2019 (18,459) to 2022 (27,747) — consistent YoY growth.")

## 6. Customer Review Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total reviews by category
review_by_cat = df.groupby('Category')['Reviews'].sum().sort_values(ascending=True)
axes[0].barh(review_by_cat.index, review_by_cat.values / 1e6,
             color=sns.color_palette('Greens_r', len(review_by_cat)))
axes[0].set_title('Total Reviews by Category (Millions)')
axes[0].set_xlabel('Total Reviews (M)')
for bar, val in zip(axes[0].patches, review_by_cat.values):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val/1e6:.1f}M', va='center', fontsize=9)

# Price vs Reviews scatter (sample 2000 points)
sample = df.sample(2000, random_state=42)
axes[1].scatter(sample['Price'], sample['Reviews'],
                alpha=0.3, s=15, color='#6baed6')
axes[1].set_title('Price vs Reviews (Sample of 2,000)')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_xlim(0, 500)
axes[1].set_ylim(0, 150000)

plt.tight_layout()
plt.savefig('chart3_reviews_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Insight: Men Shoes and Cameras dominate review volumes. Lower-priced products tend to have more reviews.")

## 7. Price Segmentation Analysis

In [ ]:
seg_cat = df.groupby(['Category', 'Price_Segment']).size().unstack(fill_value=0)
seg_pct = seg_cat.div(seg_cat.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar - price segments per category
seg_pct[['Budget (<$25)', 'Mid-range ($25-$100)', 'Premium (>$100)']].plot(
    kind='bar', stacked=True, ax=axes[0],
    color=['#74c476', '#fd8d3c', '#e6550d'])
axes[0].set_title('Price Segment Distribution by Category (%)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].tick_params(axis='x', rotation=35)

# Overall price distribution (boxplot)
cat_order = df.groupby('Category')['Price'].median().sort_values().index
df_box = df[df['Price'] <= 500]
df_box.boxplot(column='Price', by='Category', ax=axes[1],
               order=cat_order, vert=False)
axes[1].set_title('Price Distribution by Category (capped $500)')
axes[1].set_xlabel('Price ($)')
plt.sca(axes[1])
plt.title('')
plt.gcf().suptitle('')

plt.tight_layout()
plt.savefig('chart4_price_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print("Insight: Toys are mostly Budget. Laptops and Cameras are Premium. Men Shoes/Clothes are mixed mid-range.")

## 8. Key Business Insights & Recommendations

| # | Insight | Recommendation |
|---|---------|---------------|
| 1 | Men Shoes (23,228 products) dominates catalog volume | Prioritize inventory & ad spend here pre-Q4 |
| 2 | Laptops avg $1,001 — highest price category | High margin opportunity — expand SKUs |
| 3 | Orders grew 50% from 2019→2022 | Platform is scaling — invest in fulfillment capacity |
| 4 | Camera category has high reviews despite medium volume | Strong customer loyalty — good for upsell campaigns |
| 5 | Toys are mostly budget (<$25) with high review counts | Price-sensitive category — volume discounts could boost conversion |

---
*Analysis by: Sai Vardhan | Tools: Python, Pandas, Matplotlib, Seaborn*
